# 03 — Prepare Dataset

Loads the curated DFT dataset and generates ASE Atoms objects and dataset splits used by all downstream notebooks.

## Prerequisites / Input files
- `Fe-Mo/FullyCuratedParsedBriefSummary.pkl` — curated DFT dataset (included in repo)

## Outputs
- `Fe-Mo/Atomsobjects/Fe-Mo-POSCAR-initial-rescaled-AtomsObjects.pkl`
- `Fe-Mo/Atomsobjects/Fe-Mo-POSCAR-initial-rescaled-PymatgenStructures.pkl`
- `Fe-Mo/Atomsobjects/{delta,M,P,R}_structures.pkl` (available on Zenodo)



# Dataset Preparation

- Create atoms objects for guess structures
- output: pd.series of ase objects pr pymatgen objects
- calculate garget property for learning set: formation energies


In [1]:
from Tools.DatasetTools.Commoms import *
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer
from Tools.DatasetTools.Tools import need_to_update
from Tools.DatasetTools.SublatticeSorter import *
from pymatgen.io.ase import AseAtomsAdaptor
from mendeleev import element
import matplotlib.legend
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.brief_summary_parser import StructSummaryParser

/home/mariano/.local/micromamba/envs/datasets_ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import matplotlib

In [3]:
from matplotlib import colormaps

In [4]:

from scipy.interpolate import interp1d

In [5]:
from matplotlib.lines import Line2D
plt.rc('font', size=18)
plt.rc('xtick', labelsize=20)
plt.rc('ytick', labelsize=20)
plt.rc('lines', linewidth=3)
plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=24)
plt.rc('axes', labelsize=22)
plt.rcParams['figure.dpi'] = 300
#plt.rc('figure', figsize=(10,8))

In [6]:
fw, fh= plt.rcParams['figure.figsize'] 

In [7]:
#from mp_api.client import MPRester

# options 

In [8]:
dataset = 'Fe-Mo'  # 'Cr-Co-W' 'Fe-Mo'
cases = ['POSCAR.initial', 'POSCAR.relaxed-all']
rescale_by_atoms = True #  False]
subcases = ['rescaled',  'noscaled' ] 
Force = True
CuratedBS = os.path.join(dataset,'FullyCuratedParsedBriefSummary.pkl')

In [9]:
MP_keys = {'Fe' : 'mp-13', 'Mo' : 'mp-129'}

In [10]:
BS = pd.read_pickle(CuratedBS)

FileNotFoundError: [Errno 2] No such file or directory: 'Fe-Mo/CuratedParsedBriefSummary.pkl'

In [ ]:
bcc_bs = BS.loc[BS.index.str.contains('Fe.*bcc.*FM', regex=True)]

In [ ]:
bcc_bs

In [ ]:
try:
    Features = Featurizer(BS)
except Exception as _e:
    print(f'NOTE: BopFoxFeaturizer not available ({_e}); skipping Featurizer step.')
    Features = None

In [ ]:
BS.dropna().describe()

In [ ]:
BS.query('Phase=="C14" and Mag=="FM"')

In [ ]:
FeBS = pd.read_csv('Fe-Mo/rawdata/Fe_pv/briefsummary.dat', sep='\s+', header=None)

In [ ]:
feds = StructSummaryParser('Fe-Mo/rawdata/Fe_pv/', ListOfBriefsummary=['Fe-Mo/rawdata/Fe_pv/briefsummary.dat'], ForceMakeBS=True, )

In [ ]:
feBS = feds.BriefSummary

In [ ]:
FeC14BS = feBS[feBS.index.str.contains('C14') & feBS.index.str.contains('FM') ]#.query('Phase=="C14"')

In [ ]:
FeC14BS['Fe_pv'] = 1
FeC14BS['Mo_sv'] = 0
FeC14BS['atom_B'] = ''
FeC14BS['num_atom_B'] = 0

In [ ]:
FeC14BS['Mag'] = 'FM'

In [ ]:
FeC14BS['Phase'] = 'C14'

In [ ]:
BS  = pd.concat([BS, FeC14BS], axis=0)

In [ ]:
Features.data = BS

In [ ]:
Features.Mag = BS['Mag']

## Prepare targets 

One target still missing is formation Energy. Some Convenience functions to do this has been set

In [ ]:
if 'Fe' in dataset:
    ground_states= Features.get_ground_states_energies(force_mag_phase=('Fe_pv', 'NM', 'fcc'))
else:
    ground_states = Features.get_ground_states_energies()

In [ ]:
ground_states

As seen, at this point ground states are badly determined. They will be calculated after sanitation of dft data

In [ ]:
this_ground_states = {}
if 'Fe' in dataset:
    print('correcting for Fe')
    this_ground_states[('Fe_pv', 'FM')] = ground_states[('Fe_pv1', 'bcc', 'FM')]
    this_ground_states[('Fe_pv', 'NM')] = ground_states[('Fe_pv2', 'hcp', 'NM')]
if 'Mo' in dataset:                                                     
    this_ground_states[('Mo_sv', 'FM')] = ground_states[('Mo_sv1', 'bcc', 'FM')]
    this_ground_states[('Mo_sv', 'NM')] = ground_states[('Mo_sv1', 'bcc', 'NM')]

In [ ]:
this_other_ground_states = {}
if 'Fe' in dataset:
    print('correcting for Fe')
    this_other_ground_states[('Fe_pv', 'FM')] = ground_states[('Fe_pv1', 'bcc', 'FM')]
    this_other_ground_states[('Fe_pv', 'NM')] = ground_states[('Fe_pv4', 'fcc', 'NM')]
if 'Mo' in dataset:                                                            
    this_other_ground_states[('Mo_sv', 'FM')] = ground_states[('Mo_sv1', 'bcc', 'FM')]
    this_other_ground_states[('Mo_sv', 'NM')] = ground_states[('Mo_sv1', 'bcc', 'NM')]

In [ ]:
BS['EF_fmbcc'] = Features.get_formation_energies(this_ground_states, force_reference_energy={'Fe_pv': ground_states[('Fe_pv1', 'bcc', 'FM')]})

In [ ]:
BS['EF_nmhcp'] =  Features.get_formation_energies(this_ground_states)

In [ ]:
BS['EF_nmfcc'] =  Features.get_formation_energies(this_other_ground_states)

In [ ]:
BS.describe()

In [ ]:
fig, ax =  plt.subplots()
for target_case in ['EF_nmhcp']:
    ax = BS[target_case].plot.hist(ax=ax, label = target_case, bins=100)
ax.legend()

chech that the chemistry resetting is correct!

In [ ]:
ground_states

In [ ]:
ground_state_samples = pd.Series(ground_states).index.map(lambda t: '.'.join(t))

In [ ]:
ground_state_samples

In [ ]:
BS.query('index.str.contains("fcc|bcc|hcp") and nelem==1')[['V0', 'E0', 'B0',target_case]]

Remove FM R from training:

In [ ]:
BS = BS[~(( BS.Phase == "R" ) & (BS.Mag == 'FM'))]

In [ ]:
BS

In [ ]:
BS.to_pickle("Fe-Mo/FurtherCuratedParsedBriefSummary.pkl")

# guess structures: Define atomic volumes 

In [ ]:
#atom_volume_keys = {}
#with  MPRester("XMK2lneoOyVOQDnhEnVmyX3h4dyJuSyo") as mpr:
#    for atom, key in MP_keys.items():
#        result =mpr.summary.search(material_ids=[key])[0]
#        total_vol = result.volume
#        nsites = result.nsites
#        atom_volume_keys[atom] = total_vol/nsites
# MP Api is not compatible any more

In [ ]:
atom_volume_keys = {'Fe': 11.734084234678496, 'Mo': 15.89162790660502}

In [ ]:
try:
    Features = Featurizer(BS)
except Exception as _e:
    print(f'NOTE: BopFoxFeaturizer not available ({_e}); skipping Featurizer step.')
    Features = None

# Sort Poscar files

In [ ]:
cases

In [ ]:
for searchs, subcase in zip(cases, subcases):
#    searchs = 'POSCAR.initial'
    files = get_file_paths(dataset, searchs, csvfile=f'ListOfFiles_{searchs}.csv')

    atomsobjectslocation = os.path.join(dataset,'Atomsobjects')
    sublatticesortersfile = os.path.join(atomsobjectslocation, f'SORTERS_{searchs}.pkl')
    sublatticetagfile = os.path.join(atomsobjectslocation, f'SUBLATICETAGS_{searchs}.pkl')

    if  not os.path.exists(sublatticesortersfile): # need_to_update(sublatticesortersfile) or need_to_update(sublatticetagfile):
        SORTERS, SUBLATICETAGS = get_all_sorters_and_tags(dataset, files)
        SORTERS.to_pickle(sublatticesortersfile)
        SUBLATICETAGS.to_pickle(sublatticetagfile)
    else:
        SORTERS = pd.read_pickle(sublatticesortersfile)
        SUBLATICETAGS = pd.read_pickle(sublatticetagfile)

# Now I have to pick the atoms objects

In [ ]:
#for thiscase, (thisrescale, thissubcase) in product(case, zip(rescale_by_atoms, subcase)):
#database = f'{dataset}/**/{case}'

In [ ]:
Atoms_Objects = {}

In [ ]:
subcases

In [ ]:
for searchs, subcase in zip(cases, subcases):
    AtomsFile = os.path.join(atomsobjectslocation,f'{dataset}-{searchs}-{subcase}-AtomsObjects.pkl')
    if subcase == 'rescaled':
        booleanflag = True
    else:
        booleanflag = False
    _AtomsFileNorm = os.path.join(atomsobjectslocation,
        f'{dataset}-{searchs.replace(".", "-")}-{subcase}-AtomsObjects.pkl')
    if os.path.exists(_AtomsFileNorm):
        Atoms_Objects[searchs] = pd.read_pickle(_AtomsFileNorm)
    elif Features is not None:
        Atoms_Objects[searchs], CantMake_Atoms_Object = Features.get_atoms_object(
            database=dataset,
            rescale_by_atoms=booleanflag,
            reset_chemistry=True,
            file_filter = searchs, #'initial$',
            element_volumes=atom_volume_keys
        )
        Atoms_Objects[searchs].to_pickle(AtomsFile)
        Atoms_Objects[searchs].dropna(inplace=True)
    else:
        continue  # pkl missing and cannot recompute — skip this case
    pymatgenfile = AtomsFile.replace('AtomsObjects','PymatgenStructures')
    Pymatgen_Structures = Atoms_Objects[searchs].copy()
    if not need_to_update(pymatgenfile):
        Pymatgen_Structures = pd.read_pickle(pymatgenfile)
    else:
        Pymatgen_Structures = Atoms_Objects[searchs]['atoms'].apply(AseAtomsAdaptor.get_structure)
        Pymatgen_Structures['file'] = Atoms_Objects[searchs]['file']
        Pymatgen_Structures.to_pickle(pymatgenfile)

    accomodatewrap = Atoms_Objects[searchs].atoms.map(lambda a: a.wrap(pretty_translation=True))
    allindex = Atoms_Objects[searchs].index.intersection(BS.index)
    Atoms_Objects[searchs] = Atoms_Objects[searchs].loc[allindex]
    Atoms_Objects[searchs].to_pickle(AtomsFile)

In [ ]:
for case, AtomsObject in Atoms_Objects.items():
    difference = BS.index.difference(AtomsObject.index)
    print(case, difference)

#  visualization of some structures

In [ ]:
plt.rc('axes.spines', bottom=False, top=False, right=False, left=False)

In [ ]:
from ase.visualize.plot import plot_atoms

In [ ]:
atoms_samples = Atoms_Objects['POSCAR.initial'].atoms.sample(n=9)
fig, ax = plt.subplots(3,3, figsize = (20,20))
ax = ax.flatten()
for thisax, (thisname, thisatoms) in  zip(ax, atoms_samples.items()):
    plot_atoms(thisatoms, ax=thisax, rotation = '90x')
#    [spine.set_visible(False) for spine in thisax.spines.values()]
    thisax.set_xticks([])
    thisax.set_yticks([])
    thisax.set_title(thisname)

In [ ]:
somesigmas = Atoms_Objects['POSCAR.initial'].atoms[Atoms_Objects['POSCAR.initial'].index.str.contains('sigma')].sample(n=9)

In [ ]:
atoms_samples = Atoms_Objects['POSCAR.initial'].atoms.sample(n=9)
fig, ax = plt.subplots(3,3, figsize = (20,20))
ax = ax.flatten()
for thisax, (thisname, thisatoms) in  zip(ax, somesigmas.items()):
    plot_atoms(thisatoms, ax=thisax, rotation='90y, 90x, 45y')
    [spine.set_visible(False) for spine in thisax.spines.values()]
    thisax.set_xticks([])
    thisax.set_yticks([])
    thisax.set_title(thisname)

 For the actual visualization of the structures, we should choose one example for each structure and then draw in Vesta or Ovito for good quality figures, including coordination polyhedra etc.

# Curate Dataset to available structures 

There are still some R structures not available in data but present in briefsummaries

In [ ]:
Problems = BS.index.difference(Atoms_Objects['POSCAR.initial'].index)

In [ ]:
BS.loc[Problems]

In [ ]:
BS.dropna().describe()

In [ ]:
GoodBS = BS.loc[Atoms_Objects['POSCAR.initial'].index]

In [ ]:
GoodBS#.dropna()

 # Values as examples in the paper
 

In [ ]:
GoodBS.query('Phase.str.contains("sigma") & index.str.contains("AAABB.NM")')[['Mo_sv','EF_nmhcp']]

In [ ]:
GoodBS.query('Phase.str.contains("sigma") & index.str.contains("AABAB.NM")')[['Mo_sv','EF_nmhcp']]

In [ ]:
GoodBS.query('Phase.str.contains("sigma") & index.str.contains("AABBA.NM")')[['Mo_sv','EF_nmhcp']]

In [ ]:
GoodBS.query('Phase.str.contains("sigma") & index.str.contains("ABAAA.NM")')[['Mo_sv','EF_nmhcp']]

In [ ]:
GoodBS.query('Phase.str.contains("sigma") & index.str.contains("BAAAA.NM")')[['Mo_sv','EF_nmhcp']]

In [ ]:
FullyCuratedBSFile = os.path.join(dataset,'FullyCuratedParsedBriefSummary.pkl')

In [ ]:
GoodBS.to_pickle(FullyCuratedBSFile)

#  Final Distributions of targets

In [ ]:
plt.rc('axes.spines', bottom=True, top=True, right=True, left=True)

In [ ]:
sns.histplot(BS['B0'])

In [ ]:
##fig, ax = plt.subplots()
#plt.scatter(BS.V0, BS.E0, c=BS.B0, marker = 'o' , s = BS.B0, cmap='hot', edgecolor='k')
#cbar = plt.colorbar()
#plt.ylabel(r'$E_0$ (eV)')
#plt.xlabel(r'$V_0 (\AA^3)$')
#cbar.set_label(r'$B_0$')
##outlier_left =BS[(BS['E0']<-10) & (BS['V0']<11)].index
##plt.annotate(outlier_left[0],*BS.loc[outlier_left][['V0', 'E0']].values, fontsize=16 )

# Atomic volumes vs dft

In [ ]:
BS.query('num_atoms == 1').groupby('E0').min('E0')#['V0']#reset_index()

In [ ]:
Fe_unary = BS.query('num_atoms == 1 and Fe_pv == 1')

In [ ]:
V_GS_Fe = Fe_unary[Fe_unary.E0 == Fe_unary.E0.min()].V0

In [ ]:
V_GS_Fe

In [ ]:
Mo_unary = BS.query('num_atoms == 1 and Mo_sv == 1')

In [ ]:
V_GS_Mo = Mo_unary[Mo_unary.E0 == Mo_unary.E0.min()].V0

In [ ]:
V_GS_Mo

In [ ]:
norm = plt.Normalize(BS['Fe_pv'].min(), BS['Fe_pv'].max())
sm = plt.cm.ScalarMappable(norm=norm, cmap='rainbow')
sm.set_array([])

In [ ]:
fig = plt.figure()
axs = fig.add_axes([0.25, 0.2, 0.5, 0.7])
#axs = fig.add_axes([ 0.15, 0.2, 0.6, 0.7 ])# plt.subplots() #1, len(Atoms_Objects), figsize = (10,5), sharey=True)
#for (case, atoms_df), ax in zip(Atoms_Objects.items(), axs):
_ao_idx = Atoms_Objects['POSCAR.initial'].index.intersection(BS.index)
volumes = Atoms_Objects['POSCAR.initial'].loc[_ao_idx]['atoms'].map(lambda a: a.get_volume()/a.get_global_number_of_atoms())
_BS_aligned = BS.loc[_ao_idx]
sns.scatterplot(x=volumes.values, y=_BS_aligned['V0'].values, hue=_BS_aligned['Fe_pv'], ax=axs, palette = 'rainbow', legend='brief')
#sns.lineplot(x=BS['V0'].values, y=BS['V0'].values,ax = axs, color='k', label='x=y', legend='brief')#, hue=BS['Fe_pv'], palette='rainbow')
axs.plot([BS['V0'].min(), BS['V0'].max()],[BS['V0'].min(), BS['V0'].max()], 'k' )
axs.set_ylabel(r'DFT equilibrium volume ($ \AA /atom $)', fontsize='16')
axs.set_xlabel(r'volume of guess structure ($\AA / atom$)', fontsize='12')
axs.legend().remove()
#ax.legend(['data', 'x=y'])
handles = [
    plt.Line2D([0],[0], marker='o', linestyle='', c='k'),
    plt.Line2D([0],[0],linestyle='-', c='k')
]
labels=[
    'data',
    'DFT x=y'
]
axs.legend(handles, labels)
cbarax = fig.add_axes([0.8, 0.15, 0.02, 0.7])
cbar = axs.figure.colorbar(sm, cax=cbarax)
cbar.set_label('Fe content', fontsize=22)
fig.tight_layout()
plt.savefig(f'{dataset}/graphs/{dataset}-dftV_vs_gressV.pdf')

In [ ]:
colors = sns.color_palette('bright')
training_BS = BS.query('Phase != "bcc" & Phase != "hcp" & Phase != "fcc" & Phase != "R"')
colors

In [ ]:
phase_label = {
    'sigma': r"$\sigma$",
    'chi': r"$\chi$",
    'mu': r"$\mu$",
    'C36': 'C36',
    'C14': 'C14',
    'C15': 'C15',
    'A15': 'A15',
    'R': 'R',
    'lambda': r"$\lambda$ (C14)"
}

In [ ]:
fig, ax = plt.subplots()
sns.histplot(training_BS, y='Phase', hue='Mag', multiple='stack', palette=[colors[3], colors[4]], ax = ax)
ticks = ax.get_yticklabels()
#ax.set_xticklabels(ticklabels)
fig.savefig('Fe-Mo/graphs/Fe-Mo_StackCounts_Sanitized.pdf')
print(ticks)
new_ticks = [phase_label[label._text] for label in ticks if label._text != '']
print(new_ticks)
ax.set_yticklabels(new_ticks)
fig.savefig('Fe-Mo/graphs/Fe-Mo_StackCounts_Sanitized.pdf')


# convex hulls with magnetic mixing

In [ ]:
from Tools.DatasetTools.Tools import Plotting

In [ ]:
phases = BS.Phase.unique()

In [ ]:
CHULL = {}

In [ ]:
P = Plotting()

# Mixed convex hull

In [ ]:
for phase, bs_phase in BS.groupby(by='Phase'):
    CHULL.update (P.get_convex_hulls({phase: bs_phase}, ['Fe', 'Mo'],getproperty='EF_nmhcp'))
colors = colormaps['tab20'].colors
xy = {}
color_seq = {}
#for (phase, chull), c in zip ( CHULL.items(), colors ):
for phase, chull in CHULL.items(): #, colors ):
    if phase == 'bcc':
        continue
    if phase == 'fcc':
        continue
    if phase == 'hcp':
        continue
#    if phase == 'chi':
#        continue
#    if phase == 'R':
#        continue
    bs_phase = BS[BS.Phase == phase]
    vertices = np.unique( chull.simplices[chull.good])
    xy[phase]= bs_phase.iloc[vertices][['Fe_pv', target_case]].sort_values(by='Fe_pv')
#    l = ax.plot(xy[phase]['Fe_pv'].values, xy[phase][target_case].values, label = phase_label[phase])#, color=c)
#    ax.scatter(bs_phase['Fe_pv'], bs_phase[target_case], s=100, edgecolor='k', linewidth=2)#, facecolor=c)
#    color_seq[phase] = l[0].get_color()
#ax.legend(bbox_to_anchor=(1,1))
#ax.axhline(c='k', ls='--')
#ax.set_xlabel ('$x_{Fe}$')
#ax.set_xlim([0,1])
#ax.set_ylabel('$\Delta E_F$')
from scipy.interpolate import interp1d
interpolation_y = {}
x = np.linspace(0,1,100)
for phase, this_xy in xy.items():
    interpolation = interp1d(this_xy['Fe_pv'], this_xy[target_case])
    interpolation_y[phase]=interpolation(x)
compare_chulls = pd.DataFrame.from_dict(interpolation_y)
phase_range = {}
for phase in compare_chulls:
    mask = compare_chulls[phase].le(compare_chulls.min(axis=1))
    phase_range[phase] = x[mask]
alphas = {
    'lambda': 1,
    'R':0.3,
    'sigma': 0.3,
    'mu': 0.3
}
experimental_range = {
    'sigma': [0.4,0.55],
    'R' : [0.6,0.65],
    'mu' : [0.6, 0.55],
    'lambda': [0.655, 0.66]
}
#color_seq['R'] = 'grey'
pallete_seq = sns.color_palette('tab10', len(CHULL))
color_seq = {phase: color for phase, color in zip(CHULL.keys(), pallete_seq)}
color_seq['lambda'] = '#000000'
CHULL.keys()
marker_seq = {'A15': 'p', 'C14': 'X','C15':'>', 'C36': 'o', 'R': 'h', 'bcc':'.', 'chi': 'D', 'fcc': 9, 'hcp':'<', 'mu': 'D', 'sigma': 's'}
from matplotlib.lines import Line2D
BS.query('Mag == "FM" and Phase == "A15"')


In [ ]:
color_seq

In [ ]:
texlabels = {
    'A15': '$A$15',
    'C15': '$C$15',
    'chi': '$\chi$',
    'sigma': '$\sigma$',
    'C14':'$C$14',
    'C36': '$C$36',
    'mu': '$\mu$',
    'R': '$R$'
}

In [ ]:
def plot_one_chull(phase, chull,  bs_phase, tax):
    vertices = np.unique( chull.simplices[chull.good])
    xy[phase]= bs_phase.iloc[vertices][['Fe_pv', target_case]].sort_values(by='Fe_pv')
    tax.plot(xy[phase]['Fe_pv'].values, xy[phase][target_case].values,  color=color_seq[phase]) #label = phase_label[phase],

def plot_all_chulls(thisCHULLS, thisBS, exept_phases, ax, handles=[], labels=[]):
    for phase, chull in thisCHULLS.items(): #, colors ):
        if phase in exept_phases:
            continue
        bs_phase = thisBS[thisBS.Phase == phase]
        labels.append(texlabels.get(phase))
        handles.append(Line2D([0],[0], color=color_seq[phase], linewidth=5, marker = marker_seq[phase], markersize=8, markeredgecolor='k'))
        plot_one_chull(phase, chull, bs_phase, ax)
    return handles, labels


def plot_all_points(thisBS, exceptions, ax, handles=[], labels=[], markerlinewidth=1):
    for phase, bs_phase in thisBS.groupby('Phase'):
        if phase in exceptions:
            continue
        ax.scatter(bs_phase['Fe_pv'], bs_phase[target_case], s=45, edgecolor='k',marker=marker_seq[phase], linewidth=markerlinewidth, facecolor=color_seq[phase], zorder=10, clip_on=False)
        thetexlabel = texlabels.get(phase)
        if thetexlabel not in labels:
            labels.append(texlabels.get(phase))
            handles.append(Line2D([0],[0], color=color_seq[phase], linewidth=5, marker = marker_seq[phase], markersize=8, markeredgecolor='k'))
    return handles, labels

ax_spec = [0.2, 0.1, 0.7, 0.7]

In [ ]:
fig = plt.figure(figsize=( fw*1.5,fh*1.5 ))
#ax = fig.add_axes(ax_spec)#[0.2, 0.1, 0.6, 0.8])
ax = fig.add_subplot([0.15, 0.125, 0.75, 0.7])
exceptphases = [ 'bcc', 'fcc', 'hcp', 'R' ]
#for (phase, chull), c in zip ( CHULL.items(), colors ):
handles, labels = plot_all_chulls(CHULL, BS, exceptphases, ax, handles=[], labels=[] )

handles, labels = plot_all_points(BS, exceptphases, ax, handles, labels)

#    color_seq[phase] = l[0].get_color()
for label, trange in  experimental_range.items():
    xmin = min(trange)
    xmax = max(trange)
    this_x = x[ ( x< xmax ) & ( x>xmin )]
    ax.fill_between(this_x,-0.1, 1, alpha = alphas[label], color = color_seq[label])
    ax.annotate(phase_label[label], (xmin+0.015, 0.75), fontsize=18)

ax.legend(handles, labels, bbox_to_anchor=(0.5, 1.15), loc='center', ncol=4)#, fontsize=22)# labelspacing=2)
ax.axhline(c='k', ls='--', lw=2)
ax.set_xlabel ('$x_{Fe}$')
ax.set_xlim([0,1])
ax.set_ylim([-0.1, 0.8])
ax.set_ylabel('$\Delta E_F$ (eV/at)')
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_ConvexHulls_ExpRanges.pdf')

In [ ]:
fig = plt.figure(figsize=( fw*1.5,fh*1.5 ))
#ax = fig.add_axes(ax_spec)#[0.2, 0.1, 0.6, 0.8])
ax = fig.add_subplot([0.15, 0.125, 0.75, 0.7])
#for (phase, chull), c in zip ( CHULL.items(), colors ):
exceptions = ['bcc', 'fcc', 'hcp', 'R']
handles = []
labels = []
handles, labels = plot_all_chulls(CHULL, BS, exceptphases, ax, handles=[], labels=[] )


handles, labels = plot_all_points(BS.query('Mag == "NM"'), exceptphases, ax, handles=handles, labels=labels, markerlinewidth=3)

handles, labels = plot_all_points(BS.query('Mag == "FM"'), exceptphases, ax, handles=handles, labels=labels, markerlinewidth=1)


handles.append(Line2D([],[], linestyle='None', marker='s', markersize=8, markerfacecolor='gray', markeredgewidth=3, markeredgecolor='k'))
labels.append('NM')
handles.append(Line2D([],[], linestyle='None', marker='s', markersize=8, markerfacecolor='gray', markeredgewidth=1, markeredgecolor='k'))
labels.append('FM')

#    color_seq[phase] = l[0].get_color()
for label, trange in  experimental_range.items():
    xmin = min(trange)
    xmax = max(trange)
    this_x = x[ ( x< xmax ) & ( x>xmin )]
    ax.fill_between(this_x,-0.1, 1, alpha = alphas[label], color = color_seq[label])
    ax.annotate(phase_label[label], (xmin+0.015, 0.75), fontsize=18)
ax.legend(handles, labels, bbox_to_anchor=(0.5, 1.15), loc='center', ncol=4)#, fontsize=22)#  labelspacing=1.25)
ax.axhline(c='k', ls='--')
ax.set_xlabel ('$x_{Fe}$')
ax.set_xlim([0,1])
ax.set_ylim([-0.1, 0.8])
ax.set_ylabel('$\Delta E_F$ (eV/at)')
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_ConvexHulls_ExpRanges_FMNM.pdf')

# FM convex Hull

In [ ]:
FMBS = BS.query('Mag == "FM"').sort_values('Fe_pv')
FMCHULL = {}
for phase, bs_phase in FMBS.groupby(by='Phase'):
    FMCHULL.update (P.get_convex_hulls({phase: bs_phase}, ['Fe', 'Mo'],getproperty='EF_nmhcp', viewpoint=(0.6, -20)))
colors = colormaps['tab20'].colors

In [ ]:
plt.rc('legend', fontsize=18)

In [ ]:
ax_spec

In [ ]:
experimental_range

In [ ]:
fig = plt.figure(figsize=( fw*1.5,fh*1.8 ))
ax = fig.add_subplot([0.15, 0.125, 0.75, 0.7])
#for (phase, chull), c in zip ( CHULL.items(), colors ):
exceptions = ['bcc', 'fcc', 'hcp', 'R']
handles = []
labels = []
handles, labels = plot_all_chulls(FMCHULL, FMBS, exceptphases, ax, handles=[], labels=[] )

handles, labels = plot_all_points(FMBS, exceptphases, ax, handles=handles, labels=labels, markerlinewidth=1)

#    color_seq[phase] = l[0].get_color()
for label, trange in  experimental_range.items():
    xmin = min(trange)
    xmax = max(trange)
    this_x = x[ ( x< xmax ) & ( x>xmin )]
    ax.fill_between(this_x,-0.1, 1, alpha = alphas[label], color = color_seq[label])
    ax.annotate(phase_label[label], (xmin+0.015, 0.75), fontsize=18)
ax.legend(handles, labels, bbox_to_anchor=(0.5, 1.1), loc='center', ncol=4)#, fontsize=22)#  labelspacing=1.25)
ax.axhline(c='k', ls='--', lw=1)
ax.set_xlabel ('$x_{Fe}$')
ax.set_xlim([0,1])
ax.set_ylim([-0.1, 0.8])
ax.set_ylabel('$\Delta E_F$ (eV/at)')
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_FMConvexHulls_ExpRanges_FM.pdf')

# NM convex Hull

In [ ]:
NMBS = BS.query('Mag == "NM"').sort_values('Fe_pv')
NMCHULL = {}
for phase, bs_phase in NMBS.groupby(by='Phase'):
    NMCHULL.update (P.get_convex_hulls({phase: bs_phase}, ['Fe', 'Mo'],getproperty='EF_nmhcp', viewpoint=(0.6, -20)))
colors = colormaps['tab20'].colors

In [ ]:
fig = plt.figure(figsize=( fw*1.5,fh*1.8 ))
ax = fig.add_subplot([0.15, 0.125, 0.75, 0.7])
#for (phase, chull), c in zip ( CHULL.items(), colors ):
exceptphases = ['bcc', 'fcc', 'hcp', 'R']
handles = []
labels = []
handles, labels = plot_all_chulls(NMCHULL, NMBS, exceptphases, ax, handles=[], labels=[] )

handles, labels = plot_all_points(NMBS, exceptphases, ax, handles=handles, labels=labels, markerlinewidth=1)

#    color_seq[phase] = l[0].get_color()
for label, trange in  experimental_range.items():
    xmin = min(trange)
    xmax = max(trange)
    this_x = x[ ( x< xmax ) & ( x>xmin )]
    ax.fill_between(this_x,-0.1, 1, alpha = alphas[label], color = color_seq[label])
    ax.annotate(phase_label[label], (xmin+0.015, 0.75), fontsize=18)
ax.legend(handles, labels, bbox_to_anchor=(0.5, 1.1), loc='center', ncol=4)#, fontsize=22)#  labelspacing=1.25)
ax.axhline(c='k', ls='--')
ax.set_xlabel ('$x_{Fe}$')
ax.set_xlim([0,1])
ax.set_ylim([-0.1, 0.8])
ax.set_ylabel('$\Delta E_F$ (eV/at)')
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_NMConvexHulls_ExpRanges_NM.pdf')

In [ ]:
fig = plt.figure()
ax = fig.add_axes(ax_spec)
#for (phase, chull), c in zip ( CHULL.items(), colors ):
exceptphases = ['bcc', 'fcc', 'hcp']#, 'R']
handles = []
labels = []
handles, labels = plot_all_chulls(NMCHULL, NMBS, exceptphases, ax, handles=[], labels=[] )

handles, labels = plot_all_points(NMBS, exceptphases, ax, handles=handles, labels=labels, markerlinewidth=1)

#    color_seq[phase] = l[0].get_color()
for label, trange in  experimental_range.items():
    xmin = min(trange)
    xmax = max(trange)
    this_x = x[ ( x< xmax ) & ( x>xmin )]
    ax.fill_between(this_x,-0.1, 1, alpha = alphas[label], color = color_seq[label])
    ax.annotate(phase_label[label], (xmin+0.015, 0.75))
ax.legend(handles, labels, bbox_to_anchor=(0.5, 1.15), loc='center', ncol=4, fontsize=22)#  labelspacing=1.25)
ax.axhline(c='k', ls='--')
ax.set_xlabel ('$x_{Fe}$')
ax.set_xlim([0,1])
ax.set_ylim([-0.1, 0.8])
ax.set_ylabel('$\Delta E_F$ (eV/at)')
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_NMConvexHulls_ExpRanges_NM_R.pdf')

# samples in convex hull (DFT)

In [ ]:
try:
    from IPython.display import HTML
    import ipywidgets as widgets
except ImportError:
    pass


In [ ]:
nsites = {
    'bcc' : 1,
    'fcc' : 1,
    'hcp' : 2,
    'A15': 2,
    'C15': 2,
    'C14': 3,
    'chi': 4,
    'C36': 5,
    'sigma': 5,
    'sigma_Fe_pv': 5,
    'mu': 5,
    'R': 11
}

In [ ]:
def get_config(theindex): #thephaseconfig):
    splitted = theindex.split('.')
    elements = splitted[0]
    thephaseconfig =splitted[1]
    thisnsites = nsites.get(thephaseconfig.split('-')[0])
    if '-' in thephaseconfig:
        return thephaseconfig.split('-')[-1]
    else:
        try:
            if 'Mo' in elements:
                return 'B'*thisnsites
            elif 'Fe' in elements:
                return 'A'*thisnsites
        except Exception as E:
            print(f'problems with {theindex}')

In [ ]:
get_config(BS.index[0])

In [ ]:
get_config(BS.index[-1])

In [ ]:
BS.index.map(get_config)

In [ ]:
BS['config'] = BS.index.map(get_config)

In [ ]:
try:
    Ws = []
    for phase, phasexy in xy.items():
        chull_file = f'{dataset}/results/{dataset}_{phase}_DFT_ConvexHull.tex'
        BS.loc[phasexy.index][['Mo_sv', 'B0', 'V0', 'Mag','config', 'EF_nmhcp',]].to_latex(chull_file, column_format='|'+6*'c|', float_format='%.3f')
        Ws.append(widgets.HTML(BS.loc[phasexy.index][['Fe_pv', 'B0', 'V0', 'Mag','config', 'EF_nmhcp',]].to_html()))
except (NameError, Exception):
    pass

In [ ]:
try:
    widgets.VBox(Ws)
except (NameError, Exception):
    pass

In [ ]:
xy.keys()

In [ ]:
EF = {}
CHULLBS = pd.DataFrame([])
Ws = []
for phase, phasexy in xy.items():
    thisbs = BS.loc[phasexy.index].sort_values(by='Mo_sv')#[['Mo_sv', 'Fe_pv', 'Phase', 'B0', 'V0', 'Mag','config', 'EF_nmhcp',]]
#    CHULLBS = pd.concat(
#        [
#            CHULLBS, 
#            BS.loc[phasexy.index][['Mo_sv', 'Fe_pv', 'Phase', 'B0', 'V0', 'Mag','config', 'EF_nmhcp',]].sort_values(by='Fe_pv', ascending=False)
#        ],
#        axis = 0
#    )
    for chullinx in thisbs.index:
        EF[chullinx] = {}
        theindexbase = chullinx.split('.')[:-1]
        for mg in ['NM', 'FM']:
            themgindex = '.'.join(theindexbase)+'.'+mg
            if themgindex in BS.index:
                EF[chullinx][mg] = BS.loc[themgindex]['EF_nmhcp']

In [ ]:
CHULLBS = pd.DataFrame.from_dict( EF, orient='index' )
CHULLBS = pd.concat([CHULLBS, BS.loc[CHULLBS.index, ['Fe_pv', 'Mo_sv', 'V0', 'B0', 'Phase','config', 'Mag']]], axis = 1)
CHULLBS.index = pd.MultiIndex.from_frame(CHULLBS[['Phase','config', 'Mag']])
CHULLBS.drop(columns=['Mag', 'config', 'Phase'], inplace=True)
CHULLBS.index = CHULLBS.index.droplevel( -1)

In [ ]:
full_table_chulls = CHULLBS.loc[['A15', 'C15', 'C14','chi', 'mu', 'C36','sigma','R' ]][['Mo_sv', 'Fe_pv', 'B0', 'V0', 'NM', 'FM']]

In [ ]:
full_table_chulls

In [ ]:
full_table_chulls.rename(
    columns={'Mo_sv':'$x_{Mo}$','Fe_pv': '$x_{Fe}$', 'EF_nmhcp': '$\Delta E_F$ (ev/at)', 'B0': '$B_0$ (GPa)', 'V0': r'$V_0$ ($\AA^3/at$)'},
    index={'A15': r'$A$15', 'C15': r'$C$15', 'C14': r'$C14$', 'chi': r'$\chi$', 'mu': r'$\mu$', 'sigma': r'$\sigma$', 'R': r'$R$'},
    inplace=True)

In [ ]:
formattetters = {
    '$x_{Fe}$' : '{0:.2f}'.format,
    '$x_{Mo}$':  '{0:.2f}'.format,
    '$B_0$ (GPa)':     '{0:.0f}'.format,
    '$V_0$ ($\AA^3/at$)':     '{0:.2f}'.format,
    'NM':        '{0:.3f}'.format,
    'FM':        '{0:.3f}'.format
}

In [ ]:
full_table_chulls.to_latex('Fe-Mo/results/Fe-Mo_FullDFTChullData.tex', formatters=formattetters, escape=False, multirow=True)

In [ ]:
full_table_chulls

# Magnetic vs non magnetic data: tracking missing samples

In [ ]:
FMBS

In [ ]:
samples_nm = NMBS.index.str.replace('.NM', '')

In [ ]:
samples_fm = FMBS.index.str.replace('.FM', '')

In [ ]:
missing_fm = samples_nm.difference(samples_fm)

In [ ]:
import re
import glob

In [ ]:
data_root = 'Fe-Mo/rawdata/'

In [ ]:
for missing_sample in missing_fm:
    break

In [ ]:
thedir = '-'.join(re.findall('Fe_pv|Mo_sv', missing_sample))

In [ ]:
theconfig = missing_sample.split('.')[1]

In [ ]:
sample_path = os.path.join(data_root, thedir, 'bulk', theconfig+'.FM', )

In [ ]:
sample_path

In [ ]:
relax_dirs = glob.glob(os.path.join(sample_path, '**', 'relax', '**'), recursive=True)

In [ ]:
relax_dirs

In [ ]:
# Requires raw data — skipped
pass

In [ ]:
# Requires raw data — skipped
pass

In [ ]:
# Requires raw data — skipped
pass

In [ ]:
# NOTE: relax_dirs is a list; skipped (requires raw data)
pass

In [ ]:
# Requires raw data — skipped
pass

In [ ]:
# NOTE: relax_dirs is a list; skipped (requires raw data)
pass

In [ ]:
# Requires raw data — skipped
pass

In [ ]:
# Requires raw data — skipped
pass

# Complete set of training data for R

In [ ]:
Complete_R_Train = BS.query('Phase == "R" and Mag == "NM"')[['Phase', 'Mo_sv', 'Fe_pv', 'B0', 'V0', 'Mag', 'EF_nmhcp']].sort_values(by='Fe_pv', ascending=False)

In [ ]:
Complete_R_Train['config'] = Complete_R_Train.index.map(get_config)

In [ ]:
Complete_R_Train.set_index(['Phase', 'config', 'Mag'], inplace=True)

In [ ]:
Complete_R_Train.rename(
    columns={'Mo_sv': '$x_{Mo}$','Fe_pv': '$x_{Fe}$', 'EF_nmhcp': '$\Delta E_F$ (ev/at)', 'B0': '$B_0$ (GPa)', 'V0': '$V_0$ ($\AA^3/at$)'},
    index={'A15': '$A$15', 'C15': '$C$15', 'C14': '$C14$', 'chi': '$\chi$', 'mu': '$\mu$', 'sigma': '$\sigma$', 'R': '$R$'},
    inplace=True)

In [ ]:
Complete_R_Train.to_latex('Fe-Mo/results/Fe-Mo_Rstructures_TrainingOnly.tex', float_format='%.3f')